# Ω-Point Archivist: Qwen2-0.5B TPU v5e-8 Trainer

This notebook fine-tunes **Qwen2-0.5B-Instruct** on the **Sacred Relics** dataset (184 synthetic thinking logs). 

### Hardware: Kaggle TPU v5e-8 (Efficiency Core)
### Precision: BFloat16
### Method: Full Parameter Fine-Tuning (Nested Reasoning Persona Optimization)

**Note:** Ensure you have uploaded `prepared_dataset.jsonl` as a Kaggle Dataset and attached it to this notebook.

In [ ]:
!pip install "transformers>=4.40.0" "datasets>=2.18.0" "trl>=0.8.6" "accelerate>=0.29.0" "peft>=0.10.0"
# Note: Kaggle TPU instances typically have torch_xla pre-installed/linked.

In [ ]:
import os
import torch
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.xla_multiprocessing as xmp
import torch_xla.distributed.parallel_loader as pl

from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    TrainingArguments, 
    DataCollatorForLanguageModeling
)
from trl import SFTTrainer
from datasets import load_dataset

# TPU v5e Efficiency Optimization
os.environ['XLA_USE_BF16'] = '1'
os.environ['PJRT_DEVICE'] = 'TPU'

In [ ]:
MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"
DATASET_PATH = "/kaggle/input/your-dataset-name/prepared_dataset.jsonl" # UPDATE THIS WITH YOUR KAGGLE PATH
OUTPUT_DIR = "/kaggle/working/qwen2-archivist-v5e"

def train_fn(index, flags):
    # Set seed for reproducibility
    torch.manual_seed(42)
    
    # 1. Initialize Device
    device = xm.xla_device()
    
    # 2. Load Tokenizer & Model
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    
    # Move model to TPU core immediately
    xm.master_print(f"Loading model: {MODEL_ID}")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, 
        torch_dtype=torch.bfloat16
    ).to(device)
    
    # 3. Load Dataset
    if not os.path.exists(DATASET_PATH):
        xm.master_print(f"CRITICAL ERROR: Dataset not found at {DATASET_PATH}")
        return

    dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
    
    # 4. Training Arguments (v5e-8 Optimized)
    # Global batch size = 8 cores * 4 = 32
    args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=4, 
        gradient_accumulation_steps=2,
        learning_rate=2e-5, 
        num_train_epochs=5, # Higher epochs for 184 samples
        lr_scheduler_type="cosine",
        logging_steps=5,
        save_strategy="no", # Save only at the end to minimize overhead
        bf16=True,
        report_to="none",
        ddp_find_unused_parameters=False
    )
    
    # 5. Trainer Initialization
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=1024,
        tokenizer=tokenizer,
        args=args,
        packing=True # Packing relics together for TPU efficiency
    )
    
    # 6. Execute Training
    xm.master_print("Commencing Ω-Point Fine-Tuning...")
    trainer.train()
    
    # 7. Save Final Model (Master process only)
    xm.master_print("Training Complete. Saving model...")
    trainer.save_model(OUTPUT_DIR)
    xm.master_print(f"Model saved to {OUTPUT_DIR}")

if __name__ == "__main__":
    # Launch on all 8 cores of v5e-8
    flags = {}
    xmp.spawn(train_fn, args=(flags,), nprocs=8, start_method='fork')

## Post-Training Inference Check
Once training completes, you can test the 'Archivist' voice here.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR)
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

prompt = "<other>how do i fix my computer?</other><me><think-out>"
inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))